In [5]:
import numpy as np
import joblib
import cv2

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import classification_report
from scipy.ndimage import rotate, shift

# -----------------------
# Center image
# -----------------------
def center_image(img):

    coords = np.column_stack(np.where(img > 0))

    if len(coords) == 0:
        return img

    cy, cx = coords.mean(axis=0)

    shiftx = int(np.round(img.shape[1] / 2.0 - cx))
    shifty = int(np.round(img.shape[0] / 2.0 - cy))

    M = np.float32([
        [1, 0, shiftx],
        [0, 1, shifty]
    ])

    centered = cv2.warpAffine(
        img,
        M,
        (img.shape[1], img.shape[0]),
        borderValue=0
    )

    return centered


# -----------------------
# Full preprocessing
# SAME as Streamlit app
# -----------------------
def preprocess_digit(img):

    img = img.reshape(28, 28)

    # convert to uint8
    img = (img * 255).astype(np.uint8)

    # threshold
    _, thresh = cv2.threshold(
        img,
        10,
        255,
        cv2.THRESH_BINARY
    )

    # bounding box
    coords = cv2.findNonZero(thresh)

    if coords is None:
        return img.reshape(-1).astype(np.float32) / 255.0

    x, y, w, h = cv2.boundingRect(coords)

    digit = thresh[y:y+h, x:x+w]

    # -----------------------
    # Resize while keeping ratio
    # -----------------------
    h_, w_ = digit.shape

    target_size = 18

    if h_ > w_:
        new_h = target_size
        new_w = int(w_ * (target_size / h_))
    else:
        new_w = target_size
        new_h = int(h_ * (target_size / w_))

    new_w = max(1, new_w)
    new_h = max(1, new_h)

    digit = cv2.resize(
        digit,
        (new_w, new_h),
        interpolation=cv2.INTER_LINEAR
    )

    # -----------------------
    # Pad to 28x28
    # -----------------------
    canvas = np.zeros((28, 28), dtype=np.uint8)

    x_offset = (28 - new_w) // 2
    y_offset = (28 - new_h) // 2

    canvas[
        y_offset:y_offset+new_h,
        x_offset:x_offset+new_w
    ] = digit

    digit = canvas

    # -----------------------
    # Center image
    # -----------------------
    digit = center_image(digit)

    # -----------------------
    # Blur to match MNIST style
    # -----------------------
    digit = cv2.GaussianBlur(
        digit,
        (3, 3),
        0
    )

    # normalize
    digit = digit.astype(np.float32) / 255.0

    return digit.reshape(-1)


# -----------------------
# Load MNIST
# -----------------------
print("Loading MNIST...")

mnist = fetch_openml(
    'mnist_784',
    version=1,
    as_frame=False
)

X_raw = mnist.data.astype(np.float32) / 255.0
y = mnist.target.astype(np.int64)

print("Original dataset:", X_raw.shape)

# -----------------------
# Preprocess ALL images
# -----------------------
print("Preprocessing MNIST...")

X = np.array([
    preprocess_digit(img)
    for img in X_raw
])

print("Processed dataset:", X.shape)

# -----------------------
# Split dataset
# -----------------------
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=20000,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=20000,
    random_state=42,
    stratify=y_train_val
)

# -----------------------
# Augmentation
# -----------------------
def augment(img):

    img = img.reshape(28, 28)

    # small rotation
    angle = np.random.uniform(-8, 8)

    img = rotate(
        img,
        angle,
        reshape=False,
        mode='constant',
        cval=0
    )

    # small shift
    dx = np.random.randint(-2, 3)
    dy = np.random.randint(-2, 3)

    img = shift(
        img,
        [dy, dx],
        mode='constant',
        cval=0
    )

    img = np.clip(img, 0, 1)

    # IMPORTANT:
    # reapply same preprocessing
    img = preprocess_digit(img.reshape(-1))

    return img


# -----------------------
# Build augmented dataset
# -----------------------
print("Creating augmented dataset...")

X_aug = []
y_aug = []

for i in range(len(X_train)):

    # original
    X_aug.append(X_train[i])
    y_aug.append(y_train[i])

    # augmented copy
    X_aug.append(augment(X_train[i]))
    y_aug.append(y_train[i])

X_aug = np.array(X_aug)
y_aug = np.array(y_aug)

print("Augmented shape:", X_aug.shape)

# -----------------------
# Train model
# -----------------------
print("Training ExtraTrees...")

model = ExtraTreesClassifier(
    n_estimators=100, #500,
    max_depth=10,
    random_state=42,
    min_samples_leaf=2,
    n_jobs=-1,
    #max_features='sqrt',
    #min_samples_leaf=2
)

model.fit(X_aug, y_aug)

# -----------------------
# Evaluate
# -----------------------
print("\nValidation Results:")
print(classification_report(
    y_val,
    model.predict(X_val)
))

print("\nTest Results:")
print(classification_report(
    y_test,
    model.predict(X_test)
))

# -----------------------
# Save model
# -----------------------
joblib.dump(
    model,
    "ml_extra_trees_fixed_new.pkl"
)

print("\nModel saved successfully!")

Loading MNIST...
Original dataset: (70000, 784)
Preprocessing MNIST...
Processed dataset: (70000, 784)
Creating augmented dataset...
Augmented shape: (60000, 784)
Training ExtraTrees...

Validation Results:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97      1972
           1       0.90      0.98      0.94      2251
           2       0.95      0.93      0.94      1997
           3       0.86      0.92      0.89      2040
           4       0.91      0.93      0.92      1950
           5       0.92      0.89      0.91      1804
           6       0.95      0.96      0.96      1964
           7       0.90      0.94      0.92      2084
           8       0.96      0.74      0.83      1950
           9       0.87      0.86      0.87      1988

    accuracy                           0.92     20000
   macro avg       0.92      0.91      0.91     20000
weighted avg       0.92      0.92      0.91     20000


Test Results:
              preci